# Common-support window, stacked ensemble and hourly evaluation designs

Evaluates the common-support (t + h <= 48 h) cohorts, the stacked ensemble under each landmark design and the hourly random-row, patient and hospital splits.

Data: PhysioNet/Computing in Cardiology Challenge 2019 training sets A and B (open access, CC BY 4.0), downloaded by the first code cell.

Run on Google Colab with a CPU runtime (Runtime > Run all). All results are written to `results_04.json`, which is downloaded at the end.

## Download the data

In [ ]:
import os, glob
!pip -q install awscli
!aws s3 sync --no-sign-request --only-show-errors s3://physionet-open/challenge-2019/1.0.0/training/ /content/data/
nA = len(glob.glob('/content/data/training_setA/*.psv')); nB = len(glob.glob('/content/data/training_setB/*.psv'))
print('Set A files:', nA, ' Set B files:', nB, ' (expected 20336 and 20000)')
if nA < 20336 or nB < 20000:
    print('S3 download incomplete - using wget instead (slower)')
    !wget -r -N -c -np -q https://physionet.org/files/challenge-2019/1.0.0/training/ -P /content/

## Load all ICU stays

In [ ]:
import os, glob, json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
DATA = None
for cand in ['/content/data', '/content/physionet.org/files/challenge-2019/1.0.0/training']:
    if cand and glob.glob(f'{cand}/training_setA/*.psv'):
        DATA = cand; break
assert DATA, 'PhysioNet files not found - run the download cell first'
print('Using data in', DATA)
DYN = ['HR','O2Sat','Temp','SBP','MAP','DBP','Resp','EtCO2','BaseExcess','HCO3','FiO2','pH','PaCO2','SaO2',
       'AST','BUN','Alkalinephos','Calcium','Chloride','Creatinine','Bilirubin_direct','Glucose','Lactate',
       'Magnesium','Phosphate','Potassium','Bilirubin_total','TroponinI','Hct','Hgb','PTT','WBC','Fibrinogen','Platelets']
H = 6          # prediction horizon (hours)
W = 6          # feature window (hours)
W_MIN = 6      # minimum observed hours before a landmark
K_CTRL = 10    # controls per septic stay in risk-set sampling
SEEDS = [2026, 2027, 2028, 2029, 2030]
OUT = {}

def load_set(s):
    files = sorted(glob.glob(f'{DATA}/training_set{s}/*.psv'))
    stays = {}
    t0 = time.time()
    for f in files:
        d = pd.read_csv(f, sep='|')
        pid = os.path.basename(f)[:-4]
        lab = d['SepsisLabel'].to_numpy()
        pos = np.flatnonzero(lab == 1)
        stays[pid] = dict(X=d[DYN].to_numpy(np.float32), n=len(d),
                          first=(int(pos[0]) if len(pos) else -1),
                          age=float(d['Age'].iloc[0]), sex=float(d['Gender'].iloc[0]),
                          hadm=float(d['HospAdmTime'].iloc[0]), icu0=float(d['ICULOS'].iloc[0]),
                          unit=(0 if d['Unit1'].iloc[0] == 1 else (1 if d['Unit2'].iloc[0] == 1 else 2)))
    print(f'Set {s}: {len(stays)} stays loaded in {time.time()-t0:.0f}s')
    return stays

S = {s: load_set(s) for s in ['A', 'B']}
assert len(S['A']) > 0 and len(S['B']) > 0, 'Data not found'

## Helpers

In [ ]:
def features(st, lm):
    F = np.full((len(lm), 7 * len(DYN) + 4), np.nan, np.float32)
    meta = np.zeros((len(lm), 4), np.float32)   # age, sex, unit, tp (not model features)
    for r, (pid, y, tp) in enumerate(lm.itertuples(index=False)):
        v = st[pid]; hist = v['X'][:tp + 1]; win = hist[max(0, tp - W + 1):]
        with np.errstate(all='ignore'):
            ok = ~np.isnan(win); cnt = ok.sum(0)
            mean = np.nanmean(win, 0); sd = np.nanstd(win, 0); mn = np.nanmin(win, 0); mx = np.nanmax(win, 0)
            okh = ~np.isnan(hist)
            idx = np.where(okh.any(0), hist.shape[0] - 1 - np.argmax(okh[::-1], 0), -1)
            last = np.where(idx >= 0, hist[np.maximum(idx, 0), np.arange(hist.shape[1])], np.nan)
            t = np.arange(win.shape[0], dtype=np.float32)[:, None]
            tm = np.where(ok, t, 0).sum(0) / np.maximum(cnt, 1); vm = np.where(ok, win, 0).sum(0) / np.maximum(cnt, 1)
            cov = np.where(ok, (t - tm) * (np.nan_to_num(win) - vm), 0).sum(0)
            var = np.where(ok, (t - tm) ** 2, 0).sum(0)
            slope = np.where((cnt >= 2) & (var > 0), cov / np.where(var > 0, var, 1), np.nan)
            miss = 1 - cnt / win.shape[0]
        F[r, :7 * len(DYN)] = np.concatenate([mean, sd, mn, mx, last, slope, miss])
        F[r, -4:] = [v['age'], v['sex'], v['hadm'], miss.mean()]
        meta[r] = [v['age'], v['sex'], v['unit'], tp]
    return F, meta

def hgb(seed):
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                          max_depth=6, l2_regularization=1.0, class_weight='balanced',
                                          random_state=seed)

def wq_thresholds(cal_s, cal_w, test_w, alpha):
    """Weighted conformal quantile (Tibshirani et al. 2019) for each test point; inf if not attainable."""
    o = np.argsort(cal_s); s = cal_s[o]; cw = np.cumsum(cal_w[o]); tot = cw[-1]
    need = (1 - alpha) * (tot + test_w)
    k = np.searchsorted(cw, need, side='left')
    thr = np.full(len(test_w), np.inf); ok = k < len(s); thr[ok] = s[k[ok]]
    return thr

## Cohort builders

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def septic_cases(st, h):
    rows = []
    for pid, v in st.items():
        if v['first'] < 0: continue
        tp = v['first'] + 6 - h
        if W_MIN - 1 <= tp <= v['n'] - 1: rows.append((pid, 1, tp))
    return rows

def lm_trunc(st, h, seed):
    rng = np.random.default_rng(seed); cases = septic_cases(st, h); ref = np.array([c[2] for c in cases])
    rows = list(cases)
    for pid, v in st.items():
        if v['first'] >= 0: continue
        cand = ref[ref <= v['n'] - 1]
        if len(cand): rows.append((pid, 0, int(rng.choice(cand))))
    return pd.DataFrame(rows, columns=['pid', 'y', 'tp']), 0

def lm_riskset(st, h, seed, k):
    rng = np.random.default_rng(seed); cases = septic_cases(st, h)
    neg_pid = np.array([p for p, v in st.items() if v['first'] < 0]); neg_n = np.array([st[p]['n'] for p in neg_pid])
    rows, dropped = [], 0
    for pid, _, t in cases:
        pool = np.flatnonzero(neg_n - 1 >= t + h)
        if len(pool) < k: dropped += 1; continue
        rows.append((pid, 1, t)); rows += [(neg_pid[j], 0, t) for j in rng.choice(pool, size=k, replace=False)]
    return pd.DataFrame(rows, columns=['pid', 'y', 'tp']), dropped

def build(name, h, k=None):
    out = {}
    for s in ['A', 'B']:
        seed = 2019 + h + (0 if s == 'A' else 100)
        lm, dropped = lm_trunc(S[s], h, seed) if name == 'trunc' else lm_riskset(S[s], h, seed, k)
        X, meta = features(S[s], lm)
        out[s] = dict(X=X, y=lm.y.to_numpy(), tp=lm.tp.to_numpy(), pid=lm.pid.to_numpy(), dropped=int(dropped))
    return out

## Evaluation helpers

In [ ]:
NC = len(DYN)
VIT = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2']
LAB = [c for c in DYN if c not in VIT]
blk = lambda b, cols: [b * NC + DYN.index(c) for c in cols]
FAM = {'all': None,
       'vitals_values': sum((blk(b, VIT) for b in range(6)), []),
       'labs_values': sum((blk(b, LAB) for b in range(6)), []),
       'missingness_only': blk(6, DYN) + [7 * NC + 3],
       'demographics_only': [7 * NC, 7 * NC + 1, 7 * NC + 2]}

def last(X, c): return X[:, 4 * NC + DYN.index(c)]
def sirs(X):
    z = lambda c: np.nan_to_num(last(X, c), nan=-1e9)
    zt = last(X, 'Temp')
    return ((z('HR') > 90).astype(int) + (((zt > 38) | (zt < 36)) & ~np.isnan(zt)).astype(int)
            + ((z('Resp') > 20) | ((last(X, 'PaCO2') < 32) & ~np.isnan(last(X, 'PaCO2')))).astype(int)
            + (((last(X, 'WBC') > 12) | (last(X, 'WBC') < 4)) & ~np.isnan(last(X, 'WBC'))).astype(int)).astype(float)
def qsofa(X):
    rr = last(X, 'Resp'); sbp = last(X, 'SBP')
    return (((rr >= 22) & ~np.isnan(rr)).astype(int) + ((sbp <= 100) & ~np.isnan(sbp)).astype(int)).astype(float)

def stay_split(y, pid, frac, seed):
    rng = np.random.default_rng(seed); te = np.zeros(len(y), bool)
    for lab in (0, 1):
        u = np.unique(pid[y == lab]); pick = rng.choice(u, int(round(frac * len(u))), replace=False)
        te |= np.isin(pid, pick)
    # a control stay used in several risk sets must not appear on both sides
    te = np.isin(pid, np.unique(pid[te]))
    return ~te, te

def boot_auc(y, p, pid, B=200, seed=0):
    rng = np.random.default_rng(seed); code = pd.factorize(pid)[0]; order = np.argsort(code, kind='stable')
    bounds = np.r_[0, np.cumsum(np.bincount(code))]; n_u = len(bounds) - 1; a = []
    for _ in range(B):
        u = rng.integers(0, n_u, n_u); i = np.concatenate([order[bounds[j]:bounds[j + 1]] for j in u])
        if y[i].min() != y[i].max(): a.append(roc_auc_score(y[i], p[i]))
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]

def settings(C):
    """A->B, B->A transfer and A->A, B->B within-hospital (30% of stays held out)."""
    out = {}
    for src, tgt in (('A', 'B'), ('B', 'A')):
        out[f'{src}to{tgt}'] = (C[src]['X'], C[src]['y'], C[tgt]['X'], C[tgt]['y'], C[tgt]['pid'])
    for s in ('A', 'B'):
        tr, te = stay_split(C[s]['y'], C[s]['pid'], 0.3, 2026)
        out[f'{s}to{s}'] = (C[s]['X'][tr], C[s]['y'][tr], C[s]['X'][te], C[s]['y'][te], C[s]['pid'][te])
    return out

def fit_eval(Xtr, ytr, Xte, yte, cols=None, ci=None):
    keep = (~np.isnan(Xtr)).any(0) & (~np.isnan(Xte)).any(0)
    if cols is not None:
        m_ = np.zeros(Xtr.shape[1], bool); m_[cols] = True; keep &= m_
    m = hgb(2026).fit(Xtr[:, keep], ytr); p = m.predict_proba(Xte[:, keep])[:, 1]
    r = dict(AUROC=float(roc_auc_score(yte, p)), AUPRC=float(average_precision_score(yte, p)), prev=float(yte.mean()))
    if ci is not None: r['AUROC_CI'] = boot_auc(yte, p, ci)
    return r, m, keep

## Full-window cohorts

In [ ]:
COH_T = build('trunc', H); COH_R = build('riskset', H, 10)
print('cohorts ready:', len(COH_T['A']['y']), len(COH_R['A']['y']))

## Common-support window (t + h <= 48 h)

In [ ]:
T_MAX = 48

def build_cs(name, k=None):
    """Common-support cohort: septic stays restricted to t+h <= T_MAX before matching."""
    out = {}
    for s in ['A', 'B']:
        seed = 2019 + H + (0 if s == 'A' else 100)
        st = S[s]
        sub = {p: v for p, v in st.items() if v['first'] < 0 or (v['first'] + 6 <= T_MAX)}
        lm, dropped = lm_trunc(sub, H, seed) if name == 'trunc' else lm_riskset(sub, H, seed, k)
        X, meta = features(st, lm)
        out[s] = dict(X=X, y=lm.y.to_numpy(), tp=lm.tp.to_numpy(), pid=lm.pid.to_numpy(), dropped=int(dropped))
    return out

CS = {'trunc': build_cs('trunc'), 'riskset_k10': build_cs('riskset', 10)}
CSR = {}
for name, C in CS.items():
    CSR[name] = {}
    for s in ('A', 'B'):
        y, pid = C[s]['y'], C[s]['pid']
        CSR[name][f'cohort_{s}'] = dict(rows=int(len(y)), septic=int(y.sum()), dropped=C[s]['dropped'],
                                        control_rows=int((y == 0).sum()), unique_control_stays=int(len(np.unique(pid[y == 0]))),
                                        probe_tp_AUROC=float(roc_auc_score(y, C[s]['tp'])))
    for sname, (Xtr, ytr, Xte, yte, pte) in settings(C).items():
        r, _, _ = fit_eval(Xtr, ytr, Xte, yte, ci=pte)
        r['SIRS_AUROC'] = float(roc_auc_score(yte, sirs(Xte)))
        CSR[name][sname] = r
        print(f"common-support {name:11s} {sname}: AUROC {r['AUROC']:.3f} [{r['AUROC_CI'][0]:.3f}-{r['AUROC_CI'][1]:.3f}]  SIRS {r['SIRS_AUROC']:.3f}")
for src, tgt in (('A', 'B'), ('B', 'A')):
    r = fit_eval(CS['trunc'][src]['X'], CS['trunc'][src]['y'], CS['riskset_k10'][tgt]['X'], CS['riskset_k10'][tgt]['y'])[0]
    CSR[f'cross_train_trunc_{src}_test_riskset_{tgt}'] = r
    print(f"common-support: trunc-trained on {src} scored on risk-set {tgt}: AUROC {r['AUROC']:.3f}")
OUT['common_support_48h'] = CSR

## Stacked pipeline (SMOTE + MI + stack + isotonic) under each design, A->B

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors
N_TREES = 200

def smote(X, y, seed, k=5):
    rng = np.random.default_rng(seed); Xp = X[y == 1]; n = int((y == 0).sum()) - int((y == 1).sum())
    if n <= 0: return X, y
    idx = NearestNeighbors(n_neighbors=k + 1).fit(Xp).kneighbors(Xp, return_distance=False)[:, 1:]
    i = rng.integers(0, len(Xp), n); j = idx[i, rng.integers(0, k, n)]; lam = rng.random((n, 1))
    return np.vstack([X, Xp[i] + lam * (Xp[j] - Xp[i])]), np.r_[y, np.ones(n, int)]

def bases(seed):
    return [hgb(seed).set_params(class_weight=None),
            RandomForestClassifier(n_estimators=N_TREES, min_samples_leaf=5, max_features='sqrt', n_jobs=-1, random_state=seed),
            ExtraTreesClassifier(n_estimators=N_TREES, min_samples_leaf=5, max_features='sqrt', n_jobs=-1, random_state=seed)]

def stacked_pipeline(Cs, Ct, seed=2026):
    X, y, pid = Cs['X'], Cs['y'], Cs['pid']
    # stay-level 60/20/20 split: train / isotonic / (conformal, unused here)
    rng = np.random.default_rng(seed); part = np.zeros(len(y), int)
    for lab in (0, 1):
        u = np.unique(pid[y == lab]); rng.shuffle(u); n = len(u)
        part[np.isin(pid, u[int(0.6 * n):int(0.8 * n)])] = 1; part[np.isin(pid, u[int(0.8 * n):])] = 2
    keep = (~np.isnan(X)).any(0) & (~np.isnan(Ct['X'])).any(0)
    tr = part == 0; ic = part == 1
    med = np.nanmedian(X[tr][:, keep], 0); med = np.nan_to_num(med, nan=0.0)
    f = lambda Z: np.nan_to_num(np.where(np.isnan(Z[:, keep]), med, Z[:, keep]))
    sc = StandardScaler().fit(f(X[tr])); P = lambda Z: sc.transform(f(Z))
    Xtr, Xic, Xt = P(X[tr]), P(X[ic]), P(Ct['X'])
    sel = np.argsort(-mutual_info_classif(Xtr, y[tr], random_state=seed))[:60]
    Xtr, Xic, Xt = Xtr[:, sel], Xic[:, sel], Xt[:, sel]
    ytr = y[tr]
    oof = np.zeros((len(ytr), 3))
    for a, b in StratifiedKFold(5, shuffle=True, random_state=seed).split(Xtr, ytr):
        Xa, ya = smote(Xtr[a], ytr[a], seed)
        for j, m in enumerate(bases(seed)): oof[b, j] = m.fit(Xa, ya).predict_proba(Xtr[b])[:, 1]
    meta = LogisticRegression(max_iter=1000).fit(oof, ytr)
    Xs_, ys_ = smote(Xtr, ytr, seed); ms = [m.fit(Xs_, ys_) for m in bases(seed)]
    S_ = lambda Z: meta.predict_proba(np.column_stack([m.predict_proba(Z)[:, 1] for m in ms]))[:, 1]
    iso = IsotonicRegression(out_of_bounds='clip', y_min=0, y_max=1).fit(S_(Xic), y[ic])
    p_raw = S_(Xt); p = iso.predict(p_raw)
    return dict(AUROC_raw=float(roc_auc_score(Ct['y'], p_raw)), AUROC_isotonic=float(roc_auc_score(Ct['y'], p)),
                AUPRC_raw=float(average_precision_score(Ct['y'], p_raw)),
                AUROC_CI_raw=boot_auc(Ct['y'], p_raw, Ct['pid']), prev=float(Ct['y'].mean()))

SUB = {}
for name, C in (('trunc', COH_T), ('riskset_k10', COH_R), ('riskset_k10_common_support', CS['riskset_k10'])):
    t0 = time.time()
    SUB[name] = stacked_pipeline(C['A'], C['B'])
    print(f"stacked pipeline, {name:27s} A->B: AUROC {SUB[name]['AUROC_raw']:.3f} "
          f"[{SUB[name]['AUROC_CI_raw'][0]:.3f}-{SUB[name]['AUROC_CI_raw'][1]:.3f}]  ({time.time()-t0:.0f}s)")
OUT['stacked_pipeline_AtoB'] = SUB

## Challenge-style hourly evaluation: row split vs patient split vs hospital split

In [ ]:
def hourly(st, max_rows=None):
    Xs, ys, ps, ic = [], [], [], []
    for pid, v in st.items():
        X = v['X'].astype(np.float32); n = X.shape[0]
        idx = np.where(~np.isnan(X), np.arange(n)[:, None], 0); np.maximum.accumulate(idx, axis=0, out=idx)
        locf = X[idx, np.arange(X.shape[1])]
        nmeas = np.cumsum(~np.isnan(X), 0).sum(1, keepdims=True).astype(np.float32)
        lab = np.zeros(n, int)
        if v['first'] >= 0: lab[v['first']:] = 1
        static = np.tile([v['age'], v['sex'], v['hadm']], (n, 1)).astype(np.float32)
        iculos = (np.arange(n) + 1).astype(np.float32)[:, None]
        Xs.append(np.hstack([locf, nmeas, static, iculos])); ys.append(lab); ps.append(np.full(n, pid))
    return np.vstack(Xs), np.concatenate(ys), np.concatenate(ps)

HX = {s: hourly(S[s]) for s in ('A', 'B')}
HR_ = {}
def fit_hourly(Xtr, ytr, Xte, yte, drop_iculos):
    cols = slice(0, -1) if drop_iculos else slice(None)
    m = hgb(2026).fit(Xtr[:, cols], ytr); p = m.predict_proba(Xte[:, cols])[:, 1]
    return dict(AUROC=float(roc_auc_score(yte, p)), AUPRC=float(average_precision_score(yte, p)), prev=float(yte.mean()))
XA, yA, pA = HX['A']; XB, yB, pB = HX['B']
rng = np.random.default_rng(2026)
row_te = rng.random(len(yA)) < 0.3
u = np.unique(pA); te_p = rng.choice(u, int(0.3 * len(u)), replace=False); pat_te = np.isin(pA, te_p)
for drop in (False, True):
    tag = 'without_ICULOS' if drop else 'with_ICULOS'
    HR_[f'random_row_split_A_{tag}'] = fit_hourly(XA[~row_te], yA[~row_te], XA[row_te], yA[row_te], drop)
    HR_[f'patient_split_A_{tag}'] = fit_hourly(XA[~pat_te], yA[~pat_te], XA[pat_te], yA[pat_te], drop)
    HR_[f'hospital_split_AtoB_{tag}'] = fit_hourly(XA, yA, XB, yB, drop)
for k, v in HR_.items(): print(f"hourly {k:38s} AUROC {v['AUROC']:.3f}  AUPRC {v['AUPRC']:.3f}")
OUT['hourly_challenge_style'] = HR_

## Save results

In [ ]:
with open('results_04.json', 'w') as fh:
    json.dump(OUT, fh, indent=1)
print('Saved results_04.json')

In [ ]:
from google.colab import files
files.download('results_04.json')